# Data modeling drills

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-drills.ipynb)

Practice for the data modeling lessons on the shop's source tables. The exercises follow the lessons: grain and
facts, star schemas, history, relationships, and loading. Each one asks you to build a table or answer a question the
way a modeling task at work would, and the expected output shows exactly what a correct answer returns.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck. Each exercise names the
lesson that teaches it.

*Last reviewed: November 2026, against DuckDB 2.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_customers` (the CRM's current view of each customer), `src_customer_changes` (the CRM's audit log of segment and city changes), `src_products` (the product catalogue), `src_orders` (one row per order, with fulfilment timestamps), `src_order_lines` (one row per product on an order), `src_payments` (one row per payment; an order can have several) and `src_promotions` and `src_order_promotions` (promotions applied to orders): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_customers (
    customer_id int primary key, name varchar, email varchar, city varchar, segment varchar, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_customer_changes (customer_id int, changed_at timestamp, city varchar, segment varchar);
insert into src_customer_changes values  -- the CRM's audit log: one row each time a customer's details changed
    (101, '2025-11-02 00:00', 'Leeds', 'smb'),
    (102, '2025-11-20 00:00', 'York',  'smb'),
    (102, '2026-02-01 00:00', 'York',  'enterprise'),
    (103, '2025-12-05 00:00', 'Bath',  'enterprise'),
    (104, '2026-01-08 00:00', 'Hull',  'smb'),
    (104, '2026-02-15 00:00', 'Leeds', 'smb'),
    (105, '2026-01-19 00:00', 'Derby', 'smb'),
    (106, '2026-02-10 00:00', 'York',  'mid-market');
create table src_products (
    sku varchar primary key, product_name varchar, category varchar, department varchar, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
create table src_orders (
    order_id int primary key, customer_id int, order_ts timestamp, channel varchar, status varchar,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku varchar, qty int, unit_price decimal(8, 2), discount decimal(8, 2),
    primary key (order_id, line_no)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
create table src_payments (payment_id varchar primary key, order_id int, paid_ts timestamp, method varchar, amount decimal(10, 2));
insert into src_payments values  -- order 1003 was paid in two parts; 1008 is unpaid and 1009 was cancelled
    ('P01', 1001, '2026-01-05 09:13', 'card',    76.99),
    ('P02', 1002, '2026-01-09 14:31', 'card',    60.00),
    ('P03', 1003, '2026-01-14 11:06', 'card',   132.50),
    ('P04', 1003, '2026-02-13 00:00', 'invoice', 100.00),
    ('P05', 1004, '2026-01-28 16:41', 'card',    34.99),
    ('P06', 1005, '2026-02-03 10:21', 'card',   114.99),
    ('P07', 1006, '2026-03-13 00:00', 'invoice', 400.00),
    ('P08', 1007, '2026-02-14 09:51', 'cash',    54.00),
    ('P09', 1010, '2026-02-27 08:31', 'cash',    33.00);
create table src_promotions (promo_code varchar primary key, promo_name varchar);
insert into src_promotions values ('WINTER10', 'Winter sale'), ('BULK', 'Bulk discount'), ('LOYAL', 'Loyalty reward');
create table src_order_promotions (order_id int, promo_code varchar, primary key (order_id, promo_code));
insert into src_order_promotions values (1002, 'WINTER10'), (1003, 'BULK'), (1003, 'LOYAL'), (1006, 'BULK');
""")
con.sql("show tables").show()

## Exercise 1: Check a declared grain

**Difficulty:** Easy · about 10 min

Taught in [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb).

A partner feed of order lines is supposed to have one row per `(order_id, line_no)`. Some rows were delivered more
than once, and one key was reused for two different products.

Return every key that appears more than once, with the number of rows (`copies`) and the number of different SKUs
on those rows (`distinct_skus`). Order by `order_id`, then `line_no`.

**Sample input**

**`feed_order_lines`**

| order_id | line_no | sku | qty |
| --- | --- | --- | --- |
| 1001 | 1 | SKU-01 | 2 |
| 1001 | 1 | SKU-01 | 2 |
| 1001 | 2 | SKU-05 | 4 |
| 1003 | 1 | SKU-02 | 10 |
| 1003 | 2 | SKU-03 | 5 |
| 1003 | 2 | SKU-04 | 5 |
| 1005 | 1 | SKU-01 | 3 |
| 1005 | 1 | SKU-01 | 3 |
| 1005 | 1 | SKU-01 | 3 |

**Expected output**

| order_id | line_no | copies | distinct_skus |
| --- | --- | --- | --- |
| 1001 | 1 | 2 | 1 |
| 1003 | 2 | 2 | 2 |
| 1005 | 1 | 3 | 1 |

In [ ]:
# Sample data for this exercise.
con.sql("""
create or replace table feed_order_lines as
select * from (values
    (1001, 1, 'SKU-01', 2), (1001, 1, 'SKU-01', 2), (1001, 2, 'SKU-05', 4),
    (1003, 1, 'SKU-02', 10), (1003, 2, 'SKU-03', 5), (1003, 2, 'SKU-04', 5),
    (1005, 1, 'SKU-01', 3), (1005, 1, 'SKU-01', 3), (1005, 1, 'SKU-01', 3)
) t(order_id, line_no, sku, qty)
""")

In [ ]:
# Write your SQL here, for example: con.sql("select ...").show()

In [ ]:
result = con.sql("""
select order_id, line_no, count(*) as copies, count(distinct sku) as distinct_skus
from feed_order_lines
group by order_id, line_no
having count(*) > 1
order by order_id, line_no
""")
result.show()

## Exercise 2: Allocate an order-level fee to lines exactly

**Difficulty:** Hard · about 25 min

Taught in [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb).

The shipping fee in `src_orders` is charged once per order. Allocate it to the order's lines in proportion to each
line's net amount (`qty * unit_price - discount`) so that the allocated fees of each order add up to the order's
fee **exactly**. Work in whole cents: give each line its proportional share rounded **down** to the cent, then
give the cents left over to the line with the largest net amount (lowest `line_no` on a tie).

Return `order_id`, `line_no`, `net_amount` and `allocated_fee` for orders 1001 and 1005, ordered by `order_id`
and `line_no`. (Order 1001 has two equal lines and a fee of 4.99: each share is 2.495, so rounding each share to
the nearest cent would allocate 5.00.)

**Expected output**

| order_id | line_no | net_amount | allocated_fee |
| --- | --- | --- | --- |
| 1001 | 1 | 36.00 | 2.50 |
| 1001 | 2 | 36.00 | 2.49 |
| 1005 | 1 | 54.00 | 2.46 |
| 1005 | 2 | 11.00 | 0.49 |
| 1005 | 3 | 45.00 | 2.04 |

In [ ]:
# Write your SQL here, for example: con.sql("select ...").show()

In [ ]:
result = con.sql("""
with lines as (
    select l.order_id, l.line_no, l.qty * l.unit_price - l.discount as net_amount,
           cast(o.shipping_fee * 100 as bigint) as fee_cents
    from src_order_lines l join src_orders o using (order_id)
    where l.order_id in (1001, 1005)
),
shares as (
    select *,
           fee_cents * cast(net_amount * 100 as bigint) // sum(cast(net_amount * 100 as bigint)) over (partition by order_id) as share_cents,
           row_number() over (partition by order_id order by net_amount desc, line_no) as size_rank
    from lines
)
select order_id, line_no, net_amount,
       cast((share_cents + case when size_rank = 1 then fee_cents - sum(share_cents) over (partition by order_id) else 0 end)
            / 100 as decimal(8, 2)) as allocated_fee
from shares
order by order_id, line_no
""")
result.show()

## Exercise 3: Daily sales with a date spine

**Difficulty:** Easy · about 10 min

Taught in [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb).

Report net sales (`qty * unit_price - discount`) of non-cancelled orders for every day from 2026-02-09 to
2026-02-15 inclusive, by order date. Days without sales must appear with `net_sales` of 0.00. Return `day`,
`weekday` (the day's name, such as `Monday`) and `net_sales`, ordered by day.

**Expected output**

| day | weekday | net_sales |
| --- | --- | --- |
| 2026-02-09 | Monday | 0.00 |
| 2026-02-10 | Tuesday | 0.00 |
| 2026-02-11 | Wednesday | 400.00 |
| 2026-02-12 | Thursday | 0.00 |
| 2026-02-13 | Friday | 0.00 |
| 2026-02-14 | Saturday | 54.00 |
| 2026-02-15 | Sunday | 0.00 |

In [ ]:
# Write your SQL here, for example: con.sql("select ...").show()

In [ ]:
result = con.sql("""
with days as (
    select cast(d as date) as day from range(date '2026-02-09', date '2026-02-16', interval 1 day) t(d)
),
sales as (
    select cast(o.order_ts as date) as day, sum(l.qty * l.unit_price - l.discount) as net_sales
    from src_orders o join src_order_lines l using (order_id)
    where o.status <> 'cancelled'
    group by all
)
select d.day, dayname(d.day) as weekday, coalesce(s.net_sales, 0) as net_sales
from days d left join sales s using (day)
order by d.day
""")
result.show()

## Exercise 4: Build a Type 2 dimension from a change log

**Difficulty:** Medium · about 15 min

Taught in [Data modeling 3: slowly changing dimensions](data-modeling-03-slowly-changing-dimensions.ipynb).

Build the history of each customer's city and segment from `src_customer_changes`. Each change row starts a version
that is valid from `changed_at` up to (not including) the customer's next change; the last version is valid until
`9999-12-31 00:00:00` and is the current one.

Return `customer_id`, `city`, `segment`, `valid_from`, `valid_to` and `is_current` for customers 102 and 104,
ordered by `customer_id` and `valid_from`.

**Sample input**

**`src_customer_changes`**

| customer_id | changed_at | city | segment |
| --- | --- | --- | --- |
| 101 | 2025-11-02 00:00:00 | Leeds | smb |
| 102 | 2025-11-20 00:00:00 | York | smb |
| 102 | 2026-02-01 00:00:00 | York | enterprise |
| 103 | 2025-12-05 00:00:00 | Bath | enterprise |
| 104 | 2026-01-08 00:00:00 | Hull | smb |
| 104 | 2026-02-15 00:00:00 | Leeds | smb |
| 105 | 2026-01-19 00:00:00 | Derby | smb |
| 106 | 2026-02-10 00:00:00 | York | mid-market |

**Expected output**

| customer_id | city | segment | valid_from | valid_to | is_current |
| --- | --- | --- | --- | --- | --- |
| 102 | York | smb | 2025-11-20 00:00:00 | 2026-02-01 00:00:00 | false |
| 102 | York | enterprise | 2026-02-01 00:00:00 | 9999-12-31 00:00:00 | true |
| 104 | Hull | smb | 2026-01-08 00:00:00 | 2026-02-15 00:00:00 | false |
| 104 | Leeds | smb | 2026-02-15 00:00:00 | 9999-12-31 00:00:00 | true |

In [ ]:
# Write your SQL here, for example: con.sql("select ...").show()

In [ ]:
result = con.sql("""
select customer_id, city, segment,
       changed_at as valid_from,
       coalesce(lead(changed_at) over w, timestamp '9999-12-31') as valid_to,
       lead(changed_at) over w is null as is_current
from src_customer_changes
where customer_id in (102, 104)
window w as (partition by customer_id order by changed_at)
order by customer_id, valid_from
""")
result.show()

## Exercise 5: Sales by the segment at the time of the order

**Difficulty:** Medium · about 15 min

Taught in [Data modeling 3: slowly changing dimensions](data-modeling-03-slowly-changing-dimensions.ipynb).

Report net sales of non-cancelled orders by month and by the customer's segment **at the time of the order**,
using the history in `src_customer_changes` (a change is effective from its `changed_at`). Return `month` (the
first day of the month, as a date), `segment` and `net_sales`, ordered by month and segment.

**Expected output**

| month | segment | net_sales |
| --- | --- | --- |
| 2026-01-01 | enterprise | 232.50 |
| 2026-01-01 | smb | 162.00 |
| 2026-02-01 | enterprise | 400.00 |
| 2026-02-01 | smb | 223.00 |

In [ ]:
# Write your SQL here, for example: con.sql("select ...").show()

In [ ]:
result = con.sql("""
with versions as (
    select customer_id, segment, changed_at as valid_from,
           coalesce(lead(changed_at) over (partition by customer_id order by changed_at), timestamp '9999-12-31') as valid_to
    from src_customer_changes
)
select date_trunc('month', o.order_ts)::date as month, v.segment,
       sum(l.qty * l.unit_price - l.discount) as net_sales
from src_orders o
join src_order_lines l using (order_id)
join versions v on v.customer_id = o.customer_id and o.order_ts >= v.valid_from and o.order_ts < v.valid_to
where o.status <> 'cancelled'
group by all
order by month, segment
""")
result.show()

## Exercise 6: Sales and payments per customer without the chasm trap

**Difficulty:** Medium · about 15 min

Taught in [Data modeling 4: relationships and double counting](data-modeling-04-relationships-and-double-counting.ipynb).

For each customer who has placed a non-cancelled order, return `customer_id`, `net_sales` (lines of their
non-cancelled orders) and `paid` (all their payments, 0 if none), then `outstanding` = net sales plus shipping fees
of those orders minus paid. Order by `customer_id`. Orders have several lines and several payments, so a single
join of lines and payments double counts.

**Expected output**

| customer_id | net_sales | paid | outstanding |
| --- | --- | --- | --- |
| 101 | 135.00 | 144.98 | 0.00 |
| 102 | 460.00 | 460.00 | 0.00 |
| 103 | 232.50 | 232.50 | 0.00 |
| 104 | 136.00 | 114.99 | 30.99 |
| 105 | 54.00 | 54.00 | 0.00 |

In [ ]:
# Write your SQL here, for example: con.sql("select ...").show()

In [ ]:
result = con.sql("""
with orders as (select order_id, customer_id, shipping_fee from src_orders where status <> 'cancelled'),
sales as (
    select o.customer_id, sum(l.qty * l.unit_price - l.discount) as net_sales
    from orders o join src_order_lines l using (order_id) group by all
),
fees as (select customer_id, sum(shipping_fee) as fees from orders group by all),
paid as (
    select o.customer_id, sum(p.amount) as paid
    from src_payments p join src_orders o using (order_id) group by all
)
select s.customer_id, s.net_sales, coalesce(p.paid, 0) as paid,
       s.net_sales + f.fees - coalesce(p.paid, 0) as outstanding
from sales s join fees f using (customer_id) left join paid p using (customer_id)
order by s.customer_id
""")
result.show()

## Exercise 7: Attribute revenue to promotions through a bridge

**Difficulty:** Medium · about 15 min

Taught in [Data modeling 4: relationships and double counting](data-modeling-04-relationships-and-double-counting.ipynb).

Orders can use several promotions (`src_order_promotions`). Split each promoted order's net revenue (sum of its
lines' `qty * unit_price - discount`) evenly across the promotions it used. Return `promo_code`, `orders` (orders
that used it), `full_credit` (the full revenue of those orders) and `allocated` (the even split, as
`decimal(10,2)`), ordered by `promo_code`. The `allocated` column must add up to the revenue of promoted orders.

**Expected output**

| promo_code | orders | full_credit | allocated |
| --- | --- | --- | --- |
| BULK | 2 | 632.50 | 516.25 |
| LOYAL | 1 | 232.50 | 116.25 |
| WINTER10 | 1 | 60.00 | 60.00 |

In [ ]:
# Write your SQL here, for example: con.sql("select ...").show()

In [ ]:
result = con.sql("""
with order_revenue as (
    select order_id, sum(qty * unit_price - discount) as revenue from src_order_lines group by order_id
),
bridge as (
    select order_id, promo_code, 1.0 / count(*) over (partition by order_id) as weight from src_order_promotions
)
select b.promo_code, count(*) as orders, sum(r.revenue) as full_credit,
       cast(sum(r.revenue * b.weight) as decimal(10, 2)) as allocated
from bridge b join order_revenue r using (order_id)
group by b.promo_code
order by b.promo_code
""")
result.show()

## Exercise 8: Month-end balances from transactions

**Difficulty:** Hard · about 25 min

Taught in [Data modeling 7: loading and late data](data-modeling-07-loading-and-late-data.ipynb).

Build a month-end receivables snapshot. For each customer and each month end (2026-01-31 and 2026-02-28), the
balance is everything billed up to and including that day (lines of non-cancelled orders plus their shipping fee,
by order date) minus everything paid up to and including that day (by payment date). Return `month_end`,
`customer_id` and `balance` for rows where the balance is not zero, ordered by `month_end` and `customer_id`.

**Expected output**

| month_end | customer_id | balance |
| --- | --- | --- |
| 2026-01-31 | 103 | 100.00 |
| 2026-02-28 | 102 | 400.00 |
| 2026-02-28 | 104 | 30.99 |

In [ ]:
# Write your SQL here, for example: con.sql("select ...").show()

In [ ]:
result = con.sql("""
with month_ends(month_end) as (values (date '2026-01-31'), (date '2026-02-28')),
billed as (
    select o.customer_id, cast(o.order_ts as date) as day,
           sum(l.qty * l.unit_price - l.discount) + any_value(o.shipping_fee) as amount
    from src_orders o join src_order_lines l using (order_id)
    where o.status <> 'cancelled'
    group by o.customer_id, o.order_id, cast(o.order_ts as date)
),
movements as (
    select customer_id, day, amount from billed
    union all
    select o.customer_id, cast(p.paid_ts as date), -p.amount from src_payments p join src_orders o using (order_id)
)
select m.month_end, v.customer_id, sum(v.amount) as balance
from month_ends m join movements v on v.day <= m.month_end
group by all
having sum(v.amount) <> 0
order by m.month_end, v.customer_id
""")
result.show()

**Next:** [Data modeling 8: physical design and performance](data-modeling-08-physical-design-and-performance.ipynb), or the [Data modeling reference](data-modeling-reference.ipynb) for a one-page summary.